# ПЗ 4.1.2. Представление данных и его влияние на ресурсы

**Дисциплина:** Системы обработки больших данных  
**Тема:** Представление данных и его влияние на ресурсы  
**Связь с ПЗ 4.1.1:** если в предыдущей работе сравнивались способы **чтения** и **загрузки** данных, то в этой работе сравниваются способы их **представления после загрузки**.

## Что должен показать результат практической работы

После выполнения работы студент должен уметь:

1. различать **dense** и **sparse** представления данных;
2. объяснять, почему текстовые и one-hot-признаки обычно образуют разрежённые матрицы;
3. сравнивать полную матрицу признаков и её компактное хранение по памяти и времени;
4. оценивать влияние размерности признакового пространства на время предобработки и обучения;
5. делать инженерный вывод о том, когда плотное представление удобно, а когда оно становится неэффективным.

## Логика практической работы

Практическая работа построена как продолжение ПЗ 4.1.1.

- В **ПЗ 4.1.1** основной вопрос был таким: *как способ чтения данных влияет на память и время выполнения?*
- В **ПЗ 4.1.2** вопрос меняется: *как способ представления уже загруженных данных влияет на память и время?*

Здесь будут исследованы три типовые ситуации:

1. **Числовые признаки** как пример компактного и привычного dense-представления.
2. **Категориальные признаки с one-hot-кодированием** как пример роста размерности и разрежённости.
3. **Текстовые признаки** как пример естественно разрежённой высокоразмерной матрицы.

В финале будет выполнен отдельный эксперимент по размерности признакового пространства.

In [1]:
from pathlib import Path
import json
import gc
import os
import time
import warnings

import numpy as np
import pandas as pd

from scipy import sparse

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report

try:
    import psutil
except ImportError:
    psutil = None

warnings.filterwarnings("ignore")

BASE_DIR = Path('.')
with open(BASE_DIR / 'variant_specs_pz_4_1_2.json', 'r', encoding='utf-8') as f:
    VARIANTS = json.load(f)

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

## 1. Выбор варианта

Установите номер варианта от **1** до **20**.  
Вариант определяет:

- число строк в числовом, категориальном и текстовом наборе;
- параметры текстовой векторизации;
- сетку значений `max_features` для эксперимента по размерности;
- акцент индивидуального аналитического вывода.

Если преподаватель назначил вариант отдельно, используйте именно его.

In [2]:
VARIANT = 1
cfg = VARIANTS[str(VARIANT)]
cfg

{'variant': 1,
 'numeric_rows': 2500,
 'events_rows': 3000,
 'text_rows': 900,
 'text_max_features': 500,
 'text_ngram_range': [1, 1],
 'text_min_df': 2,
 'dimensionality_grid': [200, 400, 600, 800],
 'focus_note': 'Сделайте основной акцент на сравнении плотной и разрежённой one-hot матрицы.'}

## 2. Вспомогательные функции

Ниже определяются функции для:

- измерения времени выполнения;
- оценки текущего потребления памяти процессом;
- оценки памяти для dense- и sparse-матриц;
- вычисления плотности матрицы признаков;
- обучения простой линейной модели и сравнения времени.

In [3]:
def rss_mb():
    if psutil is None:
        return np.nan
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 2)

def dense_memory_mb(obj):
    if isinstance(obj, pd.DataFrame):
        return obj.memory_usage(deep=True).sum() / (1024 ** 2)
    if isinstance(obj, np.ndarray):
        return obj.nbytes / (1024 ** 2)
    return np.nan

def sparse_memory_mb(x):
    if not sparse.issparse(x):
        return np.nan
    total = x.data.nbytes + x.indptr.nbytes + x.indices.nbytes
    return total / (1024 ** 2)

def density(x):
    if sparse.issparse(x):
        return x.nnz / (x.shape[0] * x.shape[1])
    if isinstance(x, pd.DataFrame):
        arr = x.to_numpy()
        return np.count_nonzero(arr) / arr.size
    if isinstance(x, np.ndarray):
        return np.count_nonzero(x) / x.size
    return np.nan

def timed_call(func, *args, **kwargs):
    rss_before = rss_mb()
    t0 = time.perf_counter()
    result = func(*args, **kwargs)
    elapsed = time.perf_counter() - t0
    rss_after = rss_mb()
    return result, elapsed, (rss_after - rss_before)

def matrix_report(name, x):
    if sparse.issparse(x):
        mem = sparse_memory_mb(x)
        kind = type(x).__name__
    elif isinstance(x, pd.DataFrame):
        mem = dense_memory_mb(x)
        kind = 'DataFrame'
    else:
        mem = dense_memory_mb(x)
        kind = type(x).__name__
    return {
        'name': name,
        'kind': kind,
        'rows': x.shape[0],
        'cols': x.shape[1],
        'density': round(float(density(x)), 6),
        'memory_mb': round(float(mem), 3)
    }

def train_linear_model(X_train, X_test, y_train, y_test):
    model = SGDClassifier(loss='log_loss', random_state=42, max_iter=1000, tol=1e-3)
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0

    t1 = time.perf_counter()
    pred = model.predict(X_test)
    pred_time = time.perf_counter() - t1

    acc = accuracy_score(y_test, pred)
    return {
        'fit_time_sec': round(fit_time, 4),
        'predict_time_sec': round(pred_time, 4),
        'accuracy': round(float(acc), 4)
    }

def cleanup(*objects):
    for obj in objects:
        try:
            del obj
        except Exception:
            pass
    gc.collect()

## 3. Описание файлов и параметров текущего варианта

In [4]:
files_overview = pd.DataFrame([
    {'file': 'numeric_profiles.csv', 'role': 'Числовые признаки (dense-базовая матрица)', 'rows_total': 9000},
    {'file': 'events_categorical.csv', 'role': 'Категориальные признаки для one-hot-кодирования', 'rows_total': 14000},
    {'file': 'messages_text.csv', 'role': 'Текстовые сообщения для векторизации', 'rows_total': 5200},
    {'file': 'variant_specs_pz_4_1_2.json', 'role': 'Описание 20 индивидуальных вариантов', 'rows_total': 20},
])
files_overview

,file,role,rows_total
0,numeric_profiles.csv,Числовые признаки (dense-базовая матрица),9000
1,events_categorical.csv,Категориальные признаки для one-hot-кодирования,14000
2,messages_text.csv,Текстовые сообщения для векторизации,5200
3,variant_specs_pz_4_1_2.json,Описание 20 индивидуальных вариантов,20


In [5]:
variant_table = pd.DataFrame([cfg]).T
variant_table.columns = ['value']
variant_table

,value
variant,1
numeric_rows,2500
events_rows,3000
text_rows,900
text_max_features,500
text_ngram_range,"[1, 1]"
text_min_df,2
dimensionality_grid,"[200, 400, 600, 800]"
focus_note,Сделайте основной акцент на сравнении плотной ...


## 4. Эксперимент 1. Числовые признаки как dense-представление

Сначала используется набор с числовыми признаками. Это контрольный эксперимент.

### Что делаем

1. Загружаем числовой набор.
2. Берём подмножество строк в соответствии с вариантом.
3. Сравниваем память:
   - исходного `DataFrame`;
   - плотной матрицы `NumPy`.
4. Обучаем линейную модель на числовом dense-представлении.

### Что нужно понять

Числовые признаки обычно хорошо ложатся в dense-матрицу, потому что большинство ячеек содержат полезные значения и не являются нулями.

In [6]:
numeric_df = pd.read_csv(BASE_DIR / 'numeric_profiles.csv').head(cfg['numeric_rows'])

numeric_features = [c for c in numeric_df.columns if c.startswith('num_')] + ['transaction_count', 'mean_amount', 'return_rate']
X_num_df = numeric_df[numeric_features].copy()
y_num = numeric_df['risk_class'].copy()

X_num_dense = X_num_df.to_numpy(dtype=np.float32)

numeric_reports = pd.DataFrame([
    matrix_report('Числовой DataFrame', X_num_df),
    matrix_report('Числовая dense-матрица NumPy', X_num_dense)
])
numeric_reports

,name,kind,rows,cols,density,memory_mb
0,Числовой DataFrame,DataFrame,2500,15,0.980693,0.286
1,Числовая dense-матрица NumPy,ndarray,2500,15,0.980693,0.143


In [7]:
X_train_num, X_test_num, y_train_num, y_test_num = train_test_split(
    X_num_dense, y_num, test_size=0.25, random_state=42, stratify=y_num
)
numeric_model_result = train_linear_model(X_train_num, X_test_num, y_train_num, y_test_num)
pd.DataFrame([numeric_model_result])

,fit_time_sec,predict_time_sec,accuracy
0,0.0454,0.0011,0.8064


### Промежуточный вывод по эксперименту 1

Ответьте письменно:

1. Почему числовой набор естественно хранить в dense-виде?
2. Есть ли смысл переводить такой набор в sparse-формат?
3. Какие достоинства и ограничения имеет dense-представление?

In [8]:
cleanup(numeric_df, X_num_df, X_num_dense, X_train_num, X_test_num, y_train_num, y_test_num, y_num)

## 5. Эксперимент 2. Категориальные признаки: dense one-hot против sparse one-hot

Теперь рассматривается набор с категориальными признаками высокой кардинальности.

### Что делаем

1. Берём несколько категориальных столбцов.
2. Строим **dense one-hot** через `pd.get_dummies()`.
3. Строим **sparse one-hot** через `OneHotEncoder(sparse_output=True)`.
4. Сравниваем:
   - число признаков;
   - плотность матрицы;
   - объём памяти;
   - время обучения модели.

### Почему это важно

One-hot-кодирование почти всегда резко увеличивает число столбцов.  
При этом каждая строка содержит единицы только в нескольких позициях, то есть матрица становится разрежённой.

In [9]:
events_df = pd.read_csv(BASE_DIR / 'events_categorical.csv').head(cfg['events_rows'])

cat_features = ['city', 'campaign', 'device', 'source', 'segment', 'page', 'region']
y_events = events_df['converted'].copy()

# Dense one-hot
(X_dense_cat, dense_time, dense_rss) = timed_call(
    pd.get_dummies,
    events_df[cat_features],
    dtype=np.uint8
)

# Sparse one-hot
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True, dtype=np.uint8)
(X_sparse_cat, sparse_time, sparse_rss) = timed_call(
    encoder.fit_transform,
    events_df[cat_features]
)

cat_reports = pd.DataFrame([
    matrix_report('Dense one-hot (pandas.get_dummies)', X_dense_cat),
    matrix_report('Sparse one-hot (OneHotEncoder)', X_sparse_cat)
])

cat_reports['build_time_sec'] = [round(dense_time, 4), round(sparse_time, 4)]
cat_reports['rss_delta_mb'] = [round(dense_rss, 4), round(sparse_rss, 4)]
cat_reports

,name,kind,rows,cols,density,memory_mb,build_time_sec,rss_delta_mb
0,Dense one-hot (pandas.get_dummies),DataFrame,3000,546,0.012821,1.562,0.0305,0.0
1,Sparse one-hot (OneHotEncoder),csr_matrix,3000,546,0.012821,0.112,0.0206,0.0


In [10]:
# Для корректного сравнения времени обучения делим данные одинаково
idx = np.arange(len(y_events))
train_idx, test_idx = train_test_split(idx, test_size=0.25, random_state=42, stratify=y_events)

X_train_dense_cat = X_dense_cat.iloc[train_idx].to_numpy(dtype=np.float32)
X_test_dense_cat = X_dense_cat.iloc[test_idx].to_numpy(dtype=np.float32)

X_train_sparse_cat = X_sparse_cat[train_idx]
X_test_sparse_cat = X_sparse_cat[test_idx]

y_train_cat = y_events.iloc[train_idx]
y_test_cat = y_events.iloc[test_idx]

cat_train_results = pd.DataFrame([
    {'representation': 'dense one-hot', **train_linear_model(X_train_dense_cat, X_test_dense_cat, y_train_cat, y_test_cat)},
    {'representation': 'sparse one-hot', **train_linear_model(X_train_sparse_cat, X_test_sparse_cat, y_train_cat, y_test_cat)},
])
cat_train_results

,representation,fit_time_sec,predict_time_sec,accuracy
0,dense one-hot,0.7161,0.0014,0.7280
1,sparse one-hot,0.0323,0.0006,0.7333


### Промежуточный вывод по эксперименту 2

Ответьте письменно:

1. Как выросло число столбцов после one-hot-кодирования?
2. Почему полученная матрица оказывается разрежённой?
3. В чём преимущество sparse one-hot по памяти?
4. Совпадают ли dense и sparse версии по качеству модели?
5. Где dense-версия удобнее, а где она уже становится неэкономичной?

In [11]:
cleanup(events_df, X_dense_cat, X_sparse_cat, X_train_dense_cat, X_test_dense_cat, X_train_sparse_cat, X_test_sparse_cat, y_events, y_train_cat, y_test_cat)

## 6. Эксперимент 3. Текстовые признаки: sparse TF-IDF против dense-представления

Текстовая векторизация — классический пример естественно разрежённой матрицы.

### Что делаем

1. Загружаем текстовый набор.
2. Векторизуем тексты с помощью `TfidfVectorizer`.
3. Получаем sparse-матрицу признаков.
4. Сравниваем её с dense-версией той же матрицы.
5. Сопоставляем память, плотность, время обучения и качество модели.

### Важная идея

Для текстов число возможных признаков велико, но каждый документ использует лишь малую их долю.  
Поэтому хранение текста в dense-виде быстро становится затратным.

In [12]:
text_df = pd.read_csv(BASE_DIR / 'messages_text.csv').head(cfg['text_rows'])

vectorizer = TfidfVectorizer(
    max_features=cfg['text_max_features'],
    ngram_range=tuple(cfg['text_ngram_range']),
    min_df=cfg['text_min_df']
)

(X_text_sparse, text_vec_time, text_vec_rss) = timed_call(
    vectorizer.fit_transform,
    text_df['text']
)

X_text_dense = X_text_sparse.astype(np.float32).toarray()
y_text = text_df['topic']

text_reports = pd.DataFrame([
    matrix_report('TF-IDF sparse', X_text_sparse),
    matrix_report('TF-IDF dense', X_text_dense)
])

text_reports['vectorization_time_sec'] = [round(text_vec_time, 4), np.nan]
text_reports['rss_delta_mb'] = [round(text_vec_rss, 4), np.nan]
text_reports

,name,kind,rows,cols,density,memory_mb,vectorization_time_sec,rss_delta_mb
0,TF-IDF sparse,csr_matrix,900,102,0.208475,0.222,0.0494,0.0
1,TF-IDF dense,ndarray,900,102,0.208475,0.350,NaN,NaN


In [13]:
X_train_text_sp, X_test_text_sp, y_train_text, y_test_text = train_test_split(
    X_text_sparse, y_text, test_size=0.25, random_state=42, stratify=y_text
)

X_train_text_de, X_test_text_de, _, _ = train_test_split(
    X_text_dense, y_text, test_size=0.25, random_state=42, stratify=y_text
)

text_train_results = pd.DataFrame([
    {'representation': 'sparse TF-IDF', **train_linear_model(X_train_text_sp, X_test_text_sp, y_train_text, y_test_text)},
    {'representation': 'dense TF-IDF', **train_linear_model(X_train_text_de, X_test_text_de, y_train_text, y_test_text)},
])
text_train_results

,representation,fit_time_sec,predict_time_sec,accuracy
0,sparse TF-IDF,0.0154,0.0005,1.0
1,dense TF-IDF,0.0301,0.0005,1.0


### Промежуточный вывод по эксперименту 3

Ответьте письменно:

1. Почему текстовая матрица признаков обычно разрежённая?
2. Насколько dense-версия увеличила объём памяти?
3. Изменилась ли точность модели при переходе к dense-форме?
4. Можно ли считать перевод sparse-матрицы в dense-представление разумным для больших текстовых корпусов?

In [14]:
cleanup(text_df, X_text_sparse, X_text_dense, X_train_text_sp, X_test_text_sp, X_train_text_de, X_test_text_de, y_text, y_train_text, y_test_text)

## 7. Эксперимент 4. Эффект размерности на время предобработки и обучения

Теперь исследуем, как рост числа признаков влияет на ресурсы.  
Для этого будем изменять параметр `max_features` в текстовом векторизаторе.

### Что нужно показать

1. как растёт размерность матрицы;
2. как меняется плотность;
3. как растёт время векторизации;
4. как меняется время обучения;
5. как меняется оценка памяти для sparse и dense форм.

In [15]:
text_df_dim = pd.read_csv(BASE_DIR / 'messages_text.csv').head(cfg['text_rows'])
y_dim = text_df_dim['topic']

dim_results = []

for mf in cfg['dimensionality_grid']:
    vect = TfidfVectorizer(
        max_features=mf,
        ngram_range=tuple(cfg['text_ngram_range']),
        min_df=cfg['text_min_df']
    )
    X_sp, vec_t, _ = timed_call(vect.fit_transform, text_df_dim['text'])

    X_train_sp, X_test_sp, y_train_sp, y_test_sp = train_test_split(
        X_sp, y_dim, test_size=0.25, random_state=42, stratify=y_dim
    )
    train_res = train_linear_model(X_train_sp, X_test_sp, y_train_sp, y_test_sp)

    dense_estimate_mb = (X_sp.shape[0] * X_sp.shape[1] * 4) / (1024 ** 2)  # float32
    dim_results.append({
        'max_features': mf,
        'rows': X_sp.shape[0],
        'cols': X_sp.shape[1],
        'density': round(float(density(X_sp)), 6),
        'sparse_memory_mb': round(float(sparse_memory_mb(X_sp)), 3),
        'dense_estimate_mb_float32': round(float(dense_estimate_mb), 3),
        'vectorization_time_sec': round(float(vec_t), 4),
        'fit_time_sec': train_res['fit_time_sec'],
        'accuracy': train_res['accuracy'],
    })

dim_results = pd.DataFrame(dim_results)
dim_results

,max_features,rows,cols,density,sparse_memory_mb,dense_estimate_mb_float32,vectorization_time_sec,fit_time_sec,accuracy
0,200,900,102,0.208475,0.222,0.35,0.0453,0.0159,1.0
1,400,900,102,0.208475,0.222,0.35,0.0358,0.0163,1.0
2,600,900,102,0.208475,0.222,0.35,0.0358,0.0148,1.0
3,800,900,102,0.208475,0.222,0.35,0.0436,0.0144,1.0


### Промежуточный вывод по эксперименту 4

Ответьте письменно:

1. Как влияет рост `max_features` на размерность матрицы?
2. Как меняется время предобработки?
3. Как меняется время обучения?
4. Почему даже при росте числа признаков sparse-представление остаётся приемлемым?
5. Что произошло бы с ресурсами при попытке хранить все такие матрицы в плотном виде?

## 8. Сводное сравнение трёх ситуаций

Ниже удобно свести вместе результаты всех базовых экспериментов.

In [16]:
summary = pd.concat([
    numeric_reports.assign(experiment='Числовые признаки')[['experiment','name','kind','rows','cols','density','memory_mb']],
    cat_reports.assign(experiment='Категориальные признаки')[['experiment','name','kind','rows','cols','density','memory_mb']],
    text_reports.assign(experiment='Текстовые признаки')[['experiment','name','kind','rows','cols','density','memory_mb']],
], ignore_index=True)

summary

,experiment,name,kind,rows,cols,density,memory_mb
0,Числовые признаки,Числовой DataFrame,DataFrame,2500,15,0.980693,0.286
1,Числовые признаки,Числовая dense-матрица NumPy,ndarray,2500,15,0.980693,0.143
2,Категориальные признаки,Dense one-hot (pandas.get_dummies),DataFrame,3000,546,0.012821,1.562
3,Категориальные признаки,Sparse one-hot (OneHotEncoder),csr_matrix,3000,546,0.012821,0.112
4,Текстовые признаки,TF-IDF sparse,csr_matrix,900,102,0.208475,0.222
5,Текстовые признаки,TF-IDF dense,ndarray,900,102,0.208475,0.350


## 9. Аналитическая часть

Подготовьте развернутые ответы по своему варианту.

1. Для каких данных dense-представление оказалось естественным и экономически оправданным?
2. В каких случаях sparse-представление дало наибольшую экономию памяти?
3. Что сильнее всего влияет на рост размерности:
   - one-hot-кодирование категорий;
   - текстовая векторизация;
   - расширение словаря признаков?
4. Как менялось время обучения при увеличении числа признаков?
5. Почему в задачах Big Data важно анализировать не только размер исходного файла, но и размер **матрицы признаков**?
6. Как результаты ПЗ 4.1.2 дополняют выводы ПЗ 4.1.1?
7. Индивидуальный аналитический акцент вашего варианта указан в следующей ячейке и должен быть обязательно раскрыт в итоговом выводе.

In [17]:
print("Индивидуальный акцент варианта:")
print(cfg['focus_note'])

Индивидуальный акцент варианта:
Сделайте основной акцент на сравнении плотной и разрежённой one-hot матрицы.


## 10. Требования к отчёту

В отчёте необходимо представить:

1. тему работы, цель и номер варианта;
2. краткое описание dense- и sparse-представлений;
3. таблицы результатов по всем экспериментам;
4. графическое или табличное сравнение изменения размерности в эксперименте 4;
5. выводы по памяти, времени и плотности матриц;
6. ответ на индивидуальный акцент варианта;
7. общий инженерный вывод: **какое представление данных следует считать предпочтительным для каждого типа исследованных данных**.

Желательно сопровождать выводы ссылкой на конкретные числовые результаты из таблиц.

## 11. Контрольные вопросы

1. Чем отличается размер исходного файла от размера построенной матрицы признаков?
2. Почему one-hot-кодирование часто приводит к разрежённой матрице?
3. Почему тексты почти всегда удобнее хранить в sparse-виде?
4. В чём различие между `dense memory` и `sparse memory`?
5. Почему рост `max_features` влияет не только на память, но и на время предобработки?
6. Можно ли заранее определить, будет ли задача склоняться к sparse-представлению? По каким признакам?
7. Как результаты этой работы подготавливают к следующей лекции о выборе алгоритмов, структур данных и инструментов?

## 12. Формула итогового вывода

Для удобства можно использовать следующую схему заключения:

> В ходе работы были сопоставлены dense- и sparse-представления данных на числовом, категориальном и текстовом наборах.  
> Установлено, что dense-представление удобно для компактных числовых признаков с высокой заполненностью, тогда как one-hot- и текстовые признаки формируют разрежённые матрицы, для которых компактное sparse-хранение существенно экономит память.  
> Показано, что рост размерности увеличивает время предобработки и обучения, особенно если попытаться использовать полную плотную матрицу признаков.  
> Следовательно, при обработке больших данных необходимо выбирать представление признаков с учётом их структуры, плотности и ожидаемой размерности.

Сравните эту формулу с собственными результатами и при необходимости скорректируйте её под ваш вариант.